# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juangriffin121/flyrank-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?**

The question is promising of good results and i'm interested in learning more of the methods which seem apropriate to work on it.   

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision**: Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring.

**Who acts on it?**: The content strategist or site owner who prioritizes work each sprint/cycle and by extension, the writers, editors, and devs who execute on whatever gets flagged.

**Cost of a wrong call?**: Rarely a dramatic failure. A decaying page left unrefreshed keeps losing rank until the fix is bigger. A high-potential page left thin never captures the traffic it could have. A winning page left unprotected breaks or drops before anyone notices. A page pruned too soon loses backlink/brand equity it can't get back.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [2]:
# Trend direction breakdown
trend_counts = df['trend_direction'].value_counts(normalize=True) * 100
print(trend_counts)

# Concrete traffic lost among declining pages
down = df[df['trend_direction'] == 'down']
print(f"Pages declining: {len(down)} ({len(down)/len(df)*100:.1f}% of dataset)")

trend_direction
down      54.206667
stable    19.873333
up        14.626667
new        7.453333
flat       3.840000
Name: proportion, dtype: float64
Pages declining: 16262 (54.2% of dataset)


In [3]:
# High demand, weak capture
median_ctr = df['ctr'].median()
high_volume_threshold = df['search_volume'].quantile(0.75)

opportunity = df[
    (df['search_volume'] >= high_volume_threshold) &
    (df['ctr'] < median_ctr)
]

print(f"High-volume, underperforming pages: {len(opportunity)} ({len(opportunity)/len(df)*100:.1f}% of dataset)")
print(f"Combined addressable search volume: {opportunity['search_volume'].sum():.0f}")

# Cross-check with thin content
thin_high_volume = opportunity[opportunity['word_count_tier'].isin(['<1000', '1000-2000'])]
print(f"Of those, thin/short content: {len(thin_high_volume)}")

High-volume, underperforming pages: 4625 (15.4% of dataset)
Combined addressable search volume: 2871600
Of those, thin/short content: 926


In [4]:
# Pareto concentration of traffic
sorted_df = df.sort_values('sessions_90d', ascending=False).reset_index(drop=True)
sorted_df['cumulative_sessions'] = sorted_df['sessions_90d'].cumsum()
total_sessions = sorted_df['sessions_90d'].sum()
sorted_df['cumulative_pct'] = sorted_df['cumulative_sessions'] / total_sessions * 100

top_10pct_count = int(len(sorted_df) * 0.10)
share_from_top_10pct = sorted_df.iloc[top_10pct_count - 1]['cumulative_pct']

print(f"Top 10% of pages ({top_10pct_count} pages) account for {share_from_top_10pct:.1f}% of total sessions")

Top 10% of pages (3000 pages) account for 65.7% of total sessions


Across ~30,000 pages, **54% are trending down**, and a Pareto check shows traffic is highly concentrated: **the top 10% of pages (3,000) drive 66% of all sessions**, meaning the pages most worth protecting are a small, identifiable set. On the growth side, **15.4% of pages (4,625) combine above-median search demand with below-median CTR**, representing **2.87M in addressable search volume** currently being under-captured and only 20% of those are explained by thin or short content, suggesting most of that gap is fixable through non-word-count levers (titles, meta, on-page relevance) rather than requiring full rewrites.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

This short exploratory data analysis suggests that there's room for improvement by flagging pages for review for refresh, expansion, protection, pruning, or monitoring.

The work will be able to flag pages to fix in order to improve performance, the model's output will only be an aid to the decision-making process, by ranking pages based on observed evidence alone, to be used by a reviewer to spend their time on the most promising pages.

This work is based on observed data only, to prove that a refresh caused a recovery, we would need an experiment which is beyond the scope of this internship.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.